In [0]:
import requests
import pandas as pds
from pyspark.sql.functions import col, current_timestamp

In [0]:
market_df=spark.sql("select MARKET_NAME from pricing_analytics.silver.daily_pricing limit 150")

In [0]:
HTTPWebSourceRootURL="https://geocoding-api.open-meteo.com/v1/search?name="
HTTPWebSourceEndURL="&count=10&language=en&format=json"

In [0]:
valid_api_endpoints=[]
for name in market_df.collect():
    HTTPWebSourceURL=f"{HTTPWebSourceRootURL}{name['MARKET_NAME']}{HTTPWebSourceEndURL}"

    api_end_point=requests.get(HTTPWebSourceURL).json()

    if isinstance(api_end_point, dict):
        valid_api_endpoints.append(api_end_point)

pandas_df=pds.DateFrame(valid_api_endpoints)

In [0]:
pandas_df=pds.DataFrame(valid_api_endpoints)
spark_df=spark.createDataFrame(pandas_df)

In [0]:
final_df=spark_df.filter(col('results').isNotNull()).withColumn("File_ingestion_date", current_timestamp())

In [0]:
final_df.write.format("json").mode("append").save("/Volumes/pricing_analytics/storageac/pricing_analytics/landing/Geo_location/")